In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
import torch
from tqdm.autonotebook import tqdm
from transformers import AutoModel, AutoTokenizer

from src.config.base import BaseConfig
from src.data.bag_of_words.config import BagOfWordsDatasetConfig, canonical_bags

device = torch.device("cuda")

In [23]:
bow_dataset_config = BagOfWordsDatasetConfig.initialize(
    snr=0.2,
    num_train_samples=100_000,
    num_val_samples=100_000,
    prompt_length=64,
    word_assignments=canonical_bags[7],
    word_decay_power=1.0
)
print(bow_dataset_config.rsq)
dataset_folder = Path(
    "artifacts/bow/dev"
)
bow_dataset_config.write_to(folder=dataset_folder)

0.03846153846153847


In [24]:
train_df = pl.read_parquet(dataset_folder / "train.parquet")
train_df["prompt"][0]

'okay wonderful bad okay okay bad bad okay terrible bad good bad great good great okay terrible terrible wonderful bad terrible good great okay good bad okay awful okay okay okay bad bad okay okay okay okay awful wonderful okay awful great okay great okay awful good bad wonderful bad good bad wonderful okay good terrible okay bad okay bad bad wonderful great bad'

In [ ]:
# pretrained_model = "Qwen/Qwen3.5-0.8B-Base"
model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompt = train_df["prompt"][0]
enc = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=True,
)

input_ids = enc["input_ids"][0]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
batch_size = 1024
all_lengths = []

prompts = pl.read_parquet(dataset_folder / "train.parquet")["prompt"].to_list()
for start in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[start : start + batch_size]

    enc = tokenizer(
        batch_prompts,
        add_special_tokens=True,
        padding=False,      # important: do not pad if you want true lengths
        truncation=False,   # important: do not truncate if you want max length
    )

    batch_lengths = [len(x) for x in enc["input_ids"]]
    all_lengths.extend(batch_lengths)
all_lengths

  0%|          | 0/98 [00:00<?, ?it/s]

[64,
 64,
 64,
 64,
 64,
 64,
 65,
 64,
 66,
 66,
 64,
 64,
 66,
 64,
 65,
 66,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 66,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 66,
 64,
 65,
 64,
 66,
 65,
 64,
 66,
 64,
 64,
 64,
 64,
 64,
 66,
 65,
 65,
 64,
 65,
 64,
 64,
 64,
 64,
 64,
 65,
 64,
 64,
 65,
 64,
 64,
 65,
 64,
 64,
 64,
 64,
 64,
 64,
 66,
 65,
 64,
 64,
 66,
 66,
 64,
 65,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 66,
 66,
 64,
 64,
 65,
 64,
 64,
 66,
 64,
 64,
 66,
 64,
 64,
 64,
 64,
 64,
 64,
 65,
 64,
 64,
 65,
 64,
 66,
 65,
 64,
 65,
 64,
 64,
 65,
 64,
 64,
 64,
 65,
 64,
 64,
 66,
 64,
 64,
 66,
 64,
 64,
 65,
 64,
 65,
 64,
 64,
 64,
 65,
 65,
 64,
 64,
 64,
 64,
 65,
 64,
 65,
 65,
 64,
 64,
 65,
 64,
 64,
 64,
 64,
 65,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 66,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 65,
 64,
 64,
 65,
 65,
 64,
 64,
 64,
 66,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 65,
 64,
